# Part 1 — Scope, Ownership, and Inputs

### Objective

Train and compare independent AKI prediction models using the versioned dataset and patient split produced by the dataset-construction notebook.

### Inputs

- Landmark or snapshot feature dataset
- Patient-level split manifest
- Feature dictionary
- Dataset quality and provenance metadata

### Rules

This notebook must not redefine cohort membership, labels, or patient splits. Input schemas and version identifiers must be validated before training.


# Part 2 — Configuration and Experiment Tracking

### Objective

Define a reproducible model run and record all training decisions.

### Required settings

- Dataset version
- Model families and search spaces
- Preprocessing rules
- Optimization metric
- Calibration option
- Random seeds
- Output paths and run identifier

### Output

A run manifest linking configuration, package versions, Git commit, dataset version, and generated artifacts.


# Part 3 — Load Data and Enforce Split Integrity

### Objective

Load the modeling data, verify the schema, and reconstruct train, validation, and test partitions from the fixed subject-level manifest.

### Rules

- Never create a new random row-level split.
- Verify that subject_id sets are mutually disjoint.
- Keep the test labels unavailable to tuning and model-selection logic.
- Report only aggregate split statistics.

### Failure condition

Stop when split overlap, missing assignments, duplicated primary keys, or incompatible dataset versions are detected.


# Part 4 — Define Features and Preprocessing

### Objective

Specify the model feature set and build preprocessing that can be fitted on training data and reapplied unchanged.

### Rules

- Exclude identifiers, timestamps, outcomes, future measurements, and split columns from model inputs.
- Fit imputers, scalers, encoders, and feature selectors without test information.
- Preserve missingness indicators when prespecified.
- Store feature order and preprocessing parameters with each model.

### Output

A reproducible preprocessing pipeline for every model family.


# Part 5 — Establish Baselines and Sanity Checks

### Objective

Create simple reference predictions and verify that the modeling setup behaves as expected before tuning complex models.

### Checks

- Outcome prevalence baseline
- Constant-probability baseline
- Feature and label alignment
- Prediction probability bounds
- Missing and infinite values
- Duplicate rows and unexpected target leakage

These checks must pass before formal model comparison begins.


# Part 6 — Train Logistic Regression

### Objective

Train Logistic Regression as the interpretable linear baseline.

### Rules

- Use training data for parameter fitting.
- Use validation data for regularization and other hyperparameter choices.
- Address class imbalance only through prespecified training strategies.
- Save the complete preprocessing and model pipeline together.

### Output

The fitted model, selected parameters, feature list, and train/validation predictions.


# Part 7 — Train Random Forest

### Objective

Train Random Forest as a nonlinear bagged-tree comparator.

### Rules

- Bound tree count, depth, leaf size, and feature sampling to control overfitting and artifact size.
- Select hyperparameters on validation performance and prespecified secondary criteria.
- Do not combine its probability with another model.

### Output

The fitted pipeline, selected parameters, and train/validation predictions.


# Part 8 — Train Boosted Trees

### Objective

Train either XGBoost or LightGBM as the boosted-tree model family.

### Rules

- Choose one primary boosted-tree implementation and record its version.
- Tune complexity, learning rate, number of estimators, row sampling, and feature sampling using training and validation data only.
- Use early stopping only with an approved non-test evaluation set.
- Do not create a weighted ensemble.

### Output

The fitted pipeline, selected parameters, and train/validation predictions.


# Part 9 — Select Models and Consider Calibration

### Objective

Compare each model family independently and lock its final configuration before test evaluation.

### Rules

- Use a prespecified primary validation metric, with calibration and operational metrics as secondary evidence.
- If probability calibration is used, fit and select it without test-set information.
- Preserve uncalibrated and calibrated probability definitions explicitly.
- Do not select a final model using test performance.

### Output

One locked configuration per model family and a documented selection decision.


# Part 10 — Generate Landmark and Dynamic Risk Predictions

### Objective

Apply the locked preprocessing and models to landmark rows and, when available, denser 1–2 hour snapshots.

### Output fields

- model_name and run_id
- subject_id and icustay_id
- snapshot_time or landmark
- predicted 48-hour AKI probability
- applicable split
- model and dataset versions

### Rules

Order trajectories chronologically and never recompute features using information after a snapshot cutoff.


# Part 11 — Validate and Save Model Artifacts

### Objective

Save complete local model artifacts with integrity checks.

### Required contents

- Preprocessing pipeline
- Fitted estimator
- Ordered feature list
- Hyperparameters
- Training metadata
- Compatible dataset schema and version
- Validation summary

### Rules

Model files, patient-level predictions, and training caches must remain local and must not be committed to GitHub.


# Part 12 — Export Predictions for Evaluation

### Objective

Write immutable prediction tables for the evaluation notebook.

### Rules

- Include identifiers only as needed for local linkage.
- Preserve labels, cutoff times, AKI onset times, landmarks, snapshots, and split names in controlled local artifacts.
- Validate one prediction per model and analysis row.
- Do not choose alert thresholds or report final test conclusions in this notebook.

### Output

Versioned prediction artifacts and aggregate training summaries.
